Aim of this script: for every row, identify whether it is the departure stop, an intermediate stop, or the terminus (final stop) of its trip — new column `depart_terminus`.

Guidance from the data provider:

> Each row should have a unique (agency, routeType, routeNumber, date, deutscheBahnStopId) tuple. For a given date, there may be a few cases where the same (agency, routeType and routeNumber) will contain records of two different trains (specifically, overlap between Austria and Germany).

> For a given (agency, routeType, routeNumber, date) tuple I would order the records by "coalesce(arrival, departure, plannedArrival, plannedDeparture)". The first record is the departure point, and the last record is the arrival point. By "coalesce" I mean a function that selects the first non-null value of its arguments, as in SQL. Of course, keeping [the overlap] in mind, this can sometimes produce incorrect results.

So a trip is identified by (`agency`, `routeType`, `routeNumber`, `date`) — we'll keep this combination around as `journey_id`. Rows get labelled:
- `depart` — first stop of the trip (by the coalesced timestamp)
- `terminus` — last stop of the trip
- `intermediate` — everything in between
- `unknown` — whenever we can't be confident in the ordering (no timestamp at all to sort by, or the trip is one of the ambiguous "two trains, one route number" cases the provider warned about)

**Language:** staying in Python/pandas — the existing cleaning pipeline is already pandas-based, ~15M rows is comfortably within pandas' range with vectorized groupby/rank (as already used in `Chuuchuu_data_cleaning.ipynb`), and keeping this in the same stack avoids a second toolchain for a one-off analysis notebook. If this logic ever needs to run outside a notebook as a recurring job, or the dataset grows an order of magnitude, a SQL engine like DuckDB (queryable directly against the parquet file, with `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)` expressing the same "first/last per trip" logic) would be a reasonable alternative — but it's not needed at the current scale.

In [ ]:
import pandas as pd
import numpy as np
import os

data_selection = "combined"

intermediate_outputs_dir = "intermediate_outputs"
data_path = f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_countries.parquet"

try: 
    data_chuuchuu.head()
except NameError:
    data_chuuchuu = pd.read_parquet(data_path)
data_chuuchuu.shape

(15000614, 26)

### Build the sort key: `coalesce(arrival, departure, plannedArrival, plannedDeparture)`

Same coalesce the provider described: the first non-null value among the four timestamp columns, per row.

Step 1: check how `arrival, departure, plannedArrival, plannedDeparture` are organized

In [2]:
data_chuuchuu[["arrival","departure","plannedArrival","plannedDeparture"]]

,arrival,departure,plannedArrival,plannedDeparture
0,2026-02-26 14:25:00+00,None,2026-02-26 14:25:00+00,2026-02-26 14:25:00+00
1,2026-02-26 14:19:00+00,2026-02-26 14:19:00+00,2026-02-26 14:19:00+00,2026-02-26 14:19:00+00
2,2026-02-26 14:07:00+00,2026-02-26 14:09:00+00,2026-02-26 14:07:00+00,2026-02-26 14:09:00+00
3,2026-02-26 14:15:00+00,2026-02-26 14:15:00+00,2026-02-26 14:15:00+00,2026-02-26 14:15:00+00
4,None,2026-02-26 14:02:00+00,2026-02-26 14:02:00+00,2026-02-26 14:02:00+00
...,...,...,...,...
15000609,2026-06-18 15:40:00+00,2026-06-18 15:55:30+00,2026-06-18 15:09:00+00,2026-06-18 15:19:00+00
15000610,2026-06-18 10:27:00+00,2026-06-18 11:56:00+00,2026-06-18 10:06:00+00,2026-06-18 11:16:00+00
15000611,2026-06-18 16:32:30+00,2026-06-18 16:45:30+00,2026-06-18 15:41:00+00,2026-06-18 15:56:00+00
15000612,2026-06-18 17:11:00+00,2026-06-18 17:20:30+00,2026-06-18 16:27:00+00,2026-06-18 16:42:00+00


In [3]:
cols = ["arrival", "departure", "plannedArrival", "plannedDeparture"]
summary = pd.DataFrame({
    "n_missing": data_chuuchuu[cols].isna().sum(),
    "pct_missing": data_chuuchuu[cols].isna().mean() * 100
})
summary

,n_missing,pct_missing
arrival,1381461,9.209363
departure,1389805,9.264987
plannedArrival,375763,2.504984
plannedDeparture,379626,2.530736


In [4]:
timestamp_cols = ["arrival", "departure", "plannedArrival", "plannedDeparture"]
for col in timestamp_cols:
    data_chuuchuu[col] = pd.to_datetime(data_chuuchuu[col], utc=True, errors="coerce")

data_chuuchuu["sort_time"] = (
    data_chuuchuu["arrival"]
    .fillna(data_chuuchuu["departure"])
    .fillna(data_chuuchuu["plannedArrival"])
    .fillna(data_chuuchuu["plannedDeparture"])
)

# Track which column was actually used
conditions = [
    data_chuuchuu["arrival"].notna(),
    data_chuuchuu["departure"].notna(),
    data_chuuchuu["plannedArrival"].notna(),
    data_chuuchuu["plannedDeparture"].notna(),
]
choices = ["arrival", "departure", "plannedArrival", "plannedDeparture"]

data_chuuchuu["sort_time_source"] = np.select(conditions, choices, default=None)

print(f"{data_chuuchuu['sort_time'].isna().sum()} rows have none of arrival/departure/plannedArrival/plannedDeparture -> can't be placed in the order")

54543 rows have none of arrival/departure/plannedArrival/plannedDeparture -> can't be placed in the order


In [5]:
data_chuuchuu["sort_time_source"].value_counts(dropna=False)

sort_time_source
arrival             13619153
departure             911920
plannedArrival        388307
None                   54543
plannedDeparture       26691
Name: count, dtype: int64

### Flag trips affected by the "two trains, one route number" ambiguity

The provider said each (`agency`, `routeType`, `routeNumber`, `date`, `deutscheBahnStopId`) tuple should be unique. A repeated `deutscheBahnStopId` within the same (`agency`, `routeType`, `routeNumber`, `date`) trip is exactly a violation of that — the same signal as the Austria/Germany overlap case they described. When that happens we can no longer trust the ordering for the whole trip (we can't tell which of the duplicate rows belongs to which physical train), so every row in that trip is marked `unknown` rather than guessed at.

In [6]:
data_chuuchuu["journey_id"] = (
    data_chuuchuu["agency"].astype(str) + "_" +
    data_chuuchuu["routeType"].astype(str) + "_" +
    data_chuuchuu["routeNumber"].astype(str) + "_" +
    data_chuuchuu["date"].astype(str)
)

dup_stop_in_trip = data_chuuchuu.duplicated(subset=["journey_id", "deutscheBahnStopId"], keep=False)
ambiguous_journey_ids = set(data_chuuchuu.loc[dup_stop_in_trip, "journey_id"])
print(f"{len(ambiguous_journey_ids)} trips contain a repeated stop id -> flagged as ambiguous")

data_chuuchuu["is_ambiguous_trip"] = data_chuuchuu["journey_id"].isin(ambiguous_journey_ids)
print(f"{data_chuuchuu['is_ambiguous_trip'].sum()} rows belong to an ambiguous trip")

0 trips contain a repeated stop id -> flagged as ambiguous
0 rows belong to an ambiguous trip


### Flag trains that may be duplicated across agencies

Separate concern from the ambiguity check above: that one catches a repeated stop *within* one agency's version of a trip. This one checks whether the same physical trip — same `routeType`, `routeNumber`, `date`, `deutscheBahnStopId` — shows up under more than one `agency`.

**Note on the requested approach:** deliberately leaving `agency` out of the id string and instead running `duplicated()` on `[journey_verificator, agency]` wouldn't actually catch cross-agency duplication — requiring `agency` to match as part of the subset means two rows can only be "duplicated" if they share the *same* agency too, which just reproduces a variant of the existing same-agency check. To find rows shared *across* agencies, we instead group by `journey_verificator` alone and count how many *distinct* agencies report it — flagging any count `> 1`.

**Caveat before treating a flag as an error:** a quick check on the full dataset shows the top flagged combinations are overwhelmingly neighboring-country agency pairs — (DB, OEBB), (DB, GTFSDE), (GTFSDE, SBB), (GTFSDE, OEBB), (FR, SBB), (DB, NS) — i.e. exactly the border pairs you'd expect if an international train is legitimately tracked by more than one national real-time feed. So a flag here isn't automatically a duplicate/error, the same way `is_ambiguous_trip` isn't — it's a checklist to review, and `GTFSDE` in particular is worth a closer look since it overlaps heavily with `DB`/`SBB`/`OEBB`, which could point to it being a broader aggregated feed rather than an independent source.

In [7]:
data_chuuchuu["journey_verificator"] = (
    data_chuuchuu["routeType"].astype(str) + "_" +
    data_chuuchuu["routeNumber"].astype(str) + "_" +
    data_chuuchuu["date"].astype(str) + "_" +
    data_chuuchuu["deutscheBahnStopId"].astype(str)
)

# count distinct agencies reporting each (routeType, routeNumber, date, stop) combination
agencies_per_verificator = data_chuuchuu.groupby("journey_verificator")["agency"].nunique()
cross_agency_verificators = set(agencies_per_verificator[agencies_per_verificator > 1].index)
print(f"{len(cross_agency_verificators)} (routeType, routeNumber, date, stop) combinations are reported by more than one agency")

data_chuuchuu["is_cross_agency_duplicate"] = data_chuuchuu["journey_verificator"].isin(cross_agency_verificators)
print(f"{data_chuuchuu['is_cross_agency_duplicate'].sum()} rows belong to one of these combinations")

# which agency pairs/groups co-occur most often, to eyeball whether this looks like real duplication
# or legitimate cross-border trains tracked by multiple national providers
flagged = data_chuuchuu[data_chuuchuu["is_cross_agency_duplicate"]]
agency_groups = flagged.groupby("journey_verificator")["agency"].apply(lambda s: tuple(sorted(s.unique())))
agency_groups.value_counts().head(20)

143060 (routeType, routeNumber, date, stop) combinations are reported by more than one agency
290730 rows belong to one of these combinations


agency
(DB, OEBB)            47078
(DB, GTFSDE)          40710
(GTFSDE, SBB)         22006
(GTFSDE, OEBB)        11065
(FR, SBB)              6068
(DB, NS)               3265
(EST, FR)              1911
(DB, GTFSDE, NS)       1353
(DB, SBB)              1139
(DB, HU)               1032
(DB, HU, OEBB)          989
(DB, DK)                969
(DB, GTFSDE, OEBB)      930
(DB, IT)                678
(DB, FR)                582
(DB, PL)                476
(DK, GTFSDE)            419
(DB, DK, GTFSDE)        385
(OEBB, SBB)             373
(HU, OEBB)              335
Name: count, dtype: int64

### How confident can we be that a cross-agency match is a real duplicate?

We can double check with time: if two agencies are reporting the *same physical stop event*, their `sort_time` (the coalesced arrival/departure timestamp) should land within a couple of minutes of each other. If instead it's a coincidental collision — two unrelated trains that happen to reuse the same `routeNumber` at the same station on the same day under different agencies — there's no reason for their times to line up at all; the gap should look essentially random (could be hours apart).

So: for every flagged `journey_verificator`, compute the spread (max − min) of `sort_time` across its rows. A small spread is strong evidence of a genuine duplicate; a large spread points to a coincidental collision instead.

In [8]:
time_spread_minutes = (
    flagged.groupby("journey_verificator")["sort_time"]
    .agg(lambda s: (s.max() - s.min()).total_seconds() / 60 if s.notna().sum() >= 2 else np.nan)
)

print(time_spread_minutes.describe())
print()
print(f"{(time_spread_minutes <= 15).mean() * 100:.1f}% of flagged combinations have every agency reporting within 15 minutes of each other")
print(f"{(time_spread_minutes > 60).mean() * 100:.1f}% have a gap of more than an hour -- likely a coincidental routeNumber collision, not a true duplicate")
print(f"{time_spread_minutes.isna().mean() * 100:.1f}% can't be checked this way (a sort_time is missing on at least one side)")

TIME_SPREAD_THRESHOLD_MINUTES = 15
likely_true_duplicate = set(time_spread_minutes[time_spread_minutes <= TIME_SPREAD_THRESHOLD_MINUTES].index)

data_chuuchuu["cross_agency_duplicate_confidence"] = "not_flagged"
data_chuuchuu.loc[data_chuuchuu["is_cross_agency_duplicate"], "cross_agency_duplicate_confidence"] = "needs_review"
data_chuuchuu.loc[data_chuuchuu["journey_verificator"].isin(likely_true_duplicate), "cross_agency_duplicate_confidence"] = "likely_true_duplicate"

data_chuuchuu["cross_agency_duplicate_confidence"].value_counts()

count    141858.000000
mean          5.661372
std          66.293633
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max        2171.500000
Name: sort_time, dtype: float64

96.0% of flagged combinations have every agency reporting within 15 minutes of each other
0.8% have a gap of more than an hour -- likely a coincidental routeNumber collision, not a true duplicate
0.8% can't be checked this way (a sort_time is missing on at least one side)


cross_agency_duplicate_confidence
not_flagged              14709884
likely_true_duplicate      278572
needs_review                12158
Name: count, dtype: int64

### Assign `depart` / `intermediate` / `terminus` by rank of `sort_time` within each trip

For rows in a non-ambiguous trip that do have a `sort_time`: rank them within their `journey_id`. Rank 1 is `depart`, the highest rank is `terminus`, everything else is `intermediate`.

A trip with only one row that has a usable `sort_time` is both "first" and "last" at once — we can't tell whether that's genuinely a single-stop trip or whether the true departure/terminus simply has no timestamp, so it's left as `unknown` rather than guessing.

Everything else — no `sort_time` at all, or part of an ambiguous trip — stays `unknown` (the column's default).

In [9]:
data_chuuchuu["depart_terminus"] = "unknown"

usable = (~data_chuuchuu["is_ambiguous_trip"]) & data_chuuchuu["sort_time"].notna()

grouped_sort_time = data_chuuchuu.groupby("journey_id", sort=False)["sort_time"]
rank = grouped_sort_time.rank(method="first")
usable_count_per_trip = grouped_sort_time.transform("count")  # count() ignores NaT, i.e. only usable rows

is_first = usable & (rank == 1)
is_last = usable & (rank == usable_count_per_trip)

data_chuuchuu.loc[is_first & ~is_last, "depart_terminus"] = "depart"
data_chuuchuu.loc[is_last & ~is_first, "depart_terminus"] = "terminus"
data_chuuchuu.loc[usable & ~is_first & ~is_last, "depart_terminus"] = "intermediate"

print(data_chuuchuu["depart_terminus"].value_counts(dropna=False))
print()
print((round((len(data_chuuchuu[data_chuuchuu["depart_terminus"]=="unknown"])/len(data_chuuchuu))*100, 2)), "% of rows have unknown depart_terminus")

depart_terminus
intermediate    12608690
terminus         1159160
depart           1159160
unknown            73604
Name: count, dtype: int64

0.49 % of rows have unknown depart_terminus


### Sanity checks

A normal trip, and (if any exist) one of the flagged ambiguous trips, printed in order so the labelling can be checked by eye.

In [10]:
cols_to_show = ["agency", "routeType", "routeNumber", "date", "stopName", "sort_time", "depart_terminus"]

sample_journey_id = data_chuuchuu.loc[data_chuuchuu["depart_terminus"] == "intermediate", "journey_id"].sample(1).iloc[0]
data_chuuchuu[data_chuuchuu["journey_id"] == sample_journey_id].sort_values("sort_time")[cols_to_show]

,agency,routeType,routeNumber,date,stopName,sort_time,depart_terminus
1013434,GTFSDE,S,5137,2026-02-27,Hauptbahnhof A1,2026-02-27 02:56:00+00:00,depart
1024963,GTFSDE,S,5137,2026-02-27,Kiel-Hassee CITTI-PARK,2026-02-27 02:58:00+00:00,intermediate
1025000,GTFSDE,S,5137,2026-02-27,Kiel-Russee,2026-02-27 03:03:00+00:00,intermediate
1024978,GTFSDE,S,5137,2026-02-27,Melsdorf,2026-02-27 03:06:00+00:00,intermediate
1024997,GTFSDE,S,5137,2026-02-27,Achterwehr,2026-02-27 03:10:00+00:00,intermediate
1024999,GTFSDE,S,5137,2026-02-27,Felde,2026-02-27 03:13:00+00:00,intermediate
1024998,GTFSDE,S,5137,2026-02-27,Bredenbek,2026-02-27 03:22:00+00:00,intermediate
1024981,GTFSDE,S,5137,2026-02-27,Schülldorf,2026-02-27 03:28:00+00:00,intermediate
1024983,GTFSDE,S,5137,2026-02-27,Rendsburg,2026-02-27 03:39:00+00:00,intermediate
1024979,GTFSDE,S,5137,2026-02-27,Owschlag,2026-02-27 03:48:00+00:00,intermediate


In [11]:
if data_chuuchuu["is_ambiguous_trip"].any():
    sample_ambiguous_journey_id = data_chuuchuu.loc[data_chuuchuu["is_ambiguous_trip"], "journey_id"].iloc[0]
    display(data_chuuchuu[data_chuuchuu["journey_id"] == sample_ambiguous_journey_id].sort_values("sort_time")[cols_to_show])
else:
    print("no ambiguous trips found in this dataset")

no ambiguous trips found in this dataset


### Invariant checks

These test the labeling *logic* itself (not the underlying data) — they should always pass regardless of what the raw data looks like, so a failure here means a bug in the code above, not a data quality issue:

1. Every resolved journey has exactly one `depart` row and exactly one `terminus` row.
2. The `depart` row's `sort_time` is the minimum, and the `terminus` row's `sort_time` is the maximum, within its journey.
3. No row is labeled `depart`/`intermediate`/`terminus` if it has no `sort_time` or belongs to an ambiguous trip — those must always stay `unknown`.

In [12]:
# work off a slim 4-column copy for these checks -- filtering/grouping the full 26-column frame
# repeatedly is needlessly memory-heavy given how many object-dtype columns it carries
check_cols = data_chuuchuu[["journey_id", "depart_terminus", "sort_time", "is_ambiguous_trip"]]

resolved = check_cols[check_cols["depart_terminus"] != "unknown"]

# 1. exactly one depart and one terminus per resolved journey
counts_per_journey = resolved.groupby("journey_id")["depart_terminus"].value_counts().unstack(fill_value=0)
bad_depart_count = counts_per_journey[counts_per_journey.get("depart", 0) != 1]
bad_terminus_count = counts_per_journey[counts_per_journey.get("terminus", 0) != 1]
print(f"journeys with != 1 depart row: {len(bad_depart_count)}")
print(f"journeys with != 1 terminus row: {len(bad_terminus_count)}")
assert len(bad_depart_count) == 0, "found a resolved journey without exactly one depart row"
assert len(bad_terminus_count) == 0, "found a resolved journey without exactly one terminus row"

# 2. depart is the min sort_time, terminus is the max sort_time, within each resolved journey
journey_time_bounds = resolved.groupby("journey_id")["sort_time"].agg(["min", "max"])
depart_rows = resolved[resolved["depart_terminus"] == "depart"].set_index("journey_id")
terminus_rows = resolved[resolved["depart_terminus"] == "terminus"].set_index("journey_id")

depart_not_min = depart_rows["sort_time"] != journey_time_bounds.loc[depart_rows.index, "min"]
terminus_not_max = terminus_rows["sort_time"] != journey_time_bounds.loc[terminus_rows.index, "max"]
print(f"depart rows NOT at the min sort_time of their journey: {depart_not_min.sum()}")
print(f"terminus rows NOT at the max sort_time of their journey: {terminus_not_max.sum()}")
assert depart_not_min.sum() == 0
assert terminus_not_max.sum() == 0

# 3. a resolved label should never occur for a row with no sort_time or in an ambiguous trip
should_be_unknown = check_cols["sort_time"].isna() | check_cols["is_ambiguous_trip"]
mislabeled = check_cols[(check_cols["depart_terminus"] != "unknown") & should_be_unknown]
print(f"rows labeled non-unknown despite no sort_time or an ambiguous trip: {len(mislabeled)}")
assert len(mislabeled) == 0

del check_cols, resolved

print("all invariant checks passed")

journeys with != 1 depart row: 0
journeys with != 1 terminus row: 0
depart rows NOT at the min sort_time of their journey: 0
terminus rows NOT at the max sort_time of their journey: 0
rows labeled non-unknown despite no sort_time or an ambiguous trip: 0
all invariant checks passed


### Domestic vs. international journeys

For a given `journey_id`, compare the `country` of its `depart` row against the `country` of its `terminus` row: if they differ, the service is `international`; if they match, it's `domestic`.

This only works for journeys where both endpoints were confidently resolved above. A journey stays `unknown` if it's `is_ambiguous_trip`, if either endpoint has no `sort_time` to rank by, or if the `country` at either endpoint itself couldn't be resolved (e.g. `UNKNOWN_COUNTRY`/`NaN`) — we don't want a missing country silently reading as "differs from" a known one and getting mislabeled `international`.

In [13]:
depart_country = data_chuuchuu.loc[data_chuuchuu["depart_terminus"] == "depart"].set_index("journey_id")["country"]
terminus_country = data_chuuchuu.loc[data_chuuchuu["depart_terminus"] == "terminus"].set_index("journey_id")["country"]

# treat UNKNOWN_COUNTRY the same as a missing country -- neither side should count as "known" for this comparison
depart_country = depart_country.replace("UNKNOWN_COUNTRY", np.nan)
terminus_country = terminus_country.replace("UNKNOWN_COUNTRY", np.nan)

journey_type = pd.Series("unknown", index=pd.Index(data_chuuchuu["journey_id"].unique(), name="journey_id"))

comparable = depart_country.index.intersection(terminus_country.index)
both_known = depart_country.loc[comparable].notna() & terminus_country.loc[comparable].notna()
comparable = comparable[both_known]

is_international = depart_country.loc[comparable] != terminus_country.loc[comparable]
journey_type.loc[comparable] = np.where(is_international, "international", "domestic")

data_chuuchuu["journey_type"] = data_chuuchuu["journey_id"].map(journey_type)

print(data_chuuchuu.drop_duplicates("journey_id")["journey_type"].value_counts(dropna=False))

journey_type
domestic         1088646
international      70514
unknown            20876
Name: count, dtype: int64


Checking the type of services

In [14]:
with pd.ExcelWriter(f"{intermediate_outputs_dir}/country_routeType_combinations.xlsx") as writer:
    data_chuuchuu[["country", "routeType"]].drop_duplicates().to_excel(writer, sheet_name="country", index=False)
    data_chuuchuu[["agency", "routeType"]].drop_duplicates().to_excel(writer, sheet_name="agency", index=False)

    # restrict the operator sheets to the June part of the dataset -- the February part has no operator
    # info at all, so including it would just flood these sheets with operator=NaN rows that don't
    # reflect an actual June-specific gap
    is_june = data_chuuchuu["date"] >= "2026-06-01"

    # don't filter out NaN operator rows here -- keeping them means a (country/agency, routeType) combo
    # that has some rows with no operator shows up with an operator=NaN row alongside its known operators,
    # so NA coverage per country/agency is visible instead of silently dropped
    data_chuuchuu.loc[is_june, ["country", "routeType", "operator"]].drop_duplicates().to_excel(writer, sheet_name="country_operator", index=False)
    data_chuuchuu.loc[is_june, ["agency", "routeType", "operator"]].drop_duplicates().to_excel(writer, sheet_name="agency_operator", index=False)

### Explore: pick a country + service type, look at a few real itineraries

Set `SELECTED_COUNTRY` and `SELECTED_ROUTE_TYPE` below (either can be `None` to drop that filter), then rerun the cell — it finds every `journey_id` with at least one stop matching both, samples a few at random, and prints each one's full itinerary in order (with `country` per stop, so you can see exactly which countries the trip passes through).

**Alternative worth considering:** if this becomes something you reach for often rather than a one-off, `ipywidgets` dropdowns (populated from `data_chuuchuu["country"].unique()` / `data_chuuchuu["routeType"].unique()`) would turn this into a proper clickable widget instead of an edit-and-rerun cell. Kept it as plain variables for now to avoid pulling in a new dependency for something you can already do by editing two lines.

In [15]:
# --- pick what to explore, then rerun this cell ---
SELECTED_COUNTRY = "None"      # set to None (or the string "None") to drop the country filter
SELECTED_ROUTE_TYPE = "FLX"       # set to None (or the string "None") to drop the service-type filter
N_JOURNEYS_TO_SAMPLE = 3

# accept the string "None" as well as the real None -- easy to type either way when editing the cell
country_filter = None if SELECTED_COUNTRY in (None, "None") else SELECTED_COUNTRY
route_type_filter = None if SELECTED_ROUTE_TYPE in (None, "None") else SELECTED_ROUTE_TYPE

mask = pd.Series(True, index=data_chuuchuu.index)
if country_filter is not None:
    mask &= data_chuuchuu["country"] == country_filter
if route_type_filter is not None:
    mask &= data_chuuchuu["routeType"] == route_type_filter

matching_journey_ids = data_chuuchuu.loc[mask, "journey_id"].unique()
print(f"{len(matching_journey_ids)} journeys match country={country_filter!r}, routeType={route_type_filter!r}")

sample_size = min(N_JOURNEYS_TO_SAMPLE, len(matching_journey_ids))
sampled_journey_ids = np.random.choice(matching_journey_ids, size=sample_size, replace=False) if sample_size else []

itinerary_cols = ["agency", "routeType", "routeNumber", "date", "stopName", "country", "sort_time", "depart_terminus", "journey_type", "operator"]

for journey_id in sampled_journey_ids:
    print(f"\n=== {journey_id} ===")
    display(data_chuuchuu[data_chuuchuu["journey_id"] == journey_id].sort_values("sort_time")[itinerary_cols])

243 journeys match country=None, routeType='FLX'

=== DB_FLX_1246_2026-06-15 ===


,agency,routeType,routeNumber,date,stopName,country,sort_time,depart_terminus,journey_type,operator
12751838,DB,FLX,1246,2026-06-15,Stuttgart Hbf,Germany,2026-06-15 14:39:00+00:00,depart,domestic,FlixTrain
12751837,DB,FLX,1246,2026-06-15,Heidelberg Hbf,Germany,2026-06-15 15:16:00+00:00,intermediate,domestic,FlixTrain
12751836,DB,FLX,1246,2026-06-15,Darmstadt Hbf,Germany,2026-06-15 15:47:00+00:00,intermediate,domestic,FlixTrain
12751843,DB,FLX,1246,2026-06-15,Frankfurt(Main)Süd,Germany,2026-06-15 16:04:00+00:00,intermediate,domestic,FlixTrain
12751842,DB,FLX,1246,2026-06-15,Erfurt Hbf,Germany,2026-06-15 18:29:00+00:00,intermediate,domestic,FlixTrain
12751841,DB,FLX,1246,2026-06-15,Halle(Saale)Hbf,Germany,2026-06-15 19:03:00+00:00,intermediate,domestic,FlixTrain
12751840,DB,FLX,1246,2026-06-15,Berlin Südkreuz,Germany,2026-06-15 20:04:00+00:00,intermediate,domestic,FlixTrain
12751839,DB,FLX,1246,2026-06-15,Berlin Hbf,Germany,2026-06-15 20:13:00+00:00,terminus,domestic,FlixTrain



=== DB_FLX_1238_2026-06-12 ===


,agency,routeType,routeNumber,date,stopName,country,sort_time,depart_terminus,journey_type,operator
9610836,DB,FLX,1238,2026-06-12,Stuttgart Hbf,Germany,2026-06-12 06:16:00+00:00,depart,domestic,FlixTrain
9610845,DB,FLX,1238,2026-06-12,Heidelberg Hbf,Germany,2026-06-12 06:59:00+00:00,intermediate,domestic,FlixTrain
9610837,DB,FLX,1238,2026-06-12,Darmstadt Hbf,Germany,2026-06-12 07:40:00+00:00,intermediate,domestic,FlixTrain
9610844,DB,FLX,1238,2026-06-12,Frankfurt(Main)Süd,Germany,2026-06-12 08:06:00+00:00,intermediate,domestic,FlixTrain
9610846,DB,FLX,1238,2026-06-12,Fulda,Germany,2026-06-12 09:13:00+00:00,intermediate,domestic,FlixTrain
9610842,DB,FLX,1238,2026-06-12,Erfurt Hbf,Germany,2026-06-12 10:23:00+00:00,intermediate,domestic,FlixTrain
9610838,DB,FLX,1238,2026-06-12,Halle(Saale)Hbf,Germany,2026-06-12 11:05:00+00:00,intermediate,domestic,FlixTrain
9610840,DB,FLX,1238,2026-06-12,Berlin Südkreuz,Germany,2026-06-12 12:13:00+00:00,intermediate,domestic,FlixTrain
9610841,DB,FLX,1238,2026-06-12,Berlin Hbf,Germany,2026-06-12 12:20:00+00:00,intermediate,domestic,FlixTrain
9610839,DB,FLX,1238,2026-06-12,Stendal Hbf,Germany,2026-06-12 13:16:00+00:00,intermediate,domestic,FlixTrain



=== DB_FLX_1239_2026-06-11 ===


,agency,routeType,routeNumber,date,stopName,country,sort_time,depart_terminus,journey_type,operator
8462408,DB,FLX,1239,2026-06-11,Hamburg Hbf,Germany,2026-06-11 07:49:00+00:00,depart,domestic,FlixTrain
8462404,DB,FLX,1239,2026-06-11,Stendal Hbf,Germany,2026-06-11 09:21:00+00:00,intermediate,domestic,FlixTrain
8462405,DB,FLX,1239,2026-06-11,Berlin-Spandau,Germany,2026-06-11 10:02:00+00:00,intermediate,domestic,FlixTrain
8462399,DB,FLX,1239,2026-06-11,Berlin Hbf,Germany,2026-06-11 10:16:00+00:00,intermediate,domestic,FlixTrain
8462398,DB,FLX,1239,2026-06-11,Berlin Südkreuz,Germany,2026-06-11 10:28:00+00:00,intermediate,domestic,FlixTrain
8462407,DB,FLX,1239,2026-06-11,Halle(Saale)Hbf,Germany,2026-06-11 11:31:00+00:00,intermediate,domestic,FlixTrain
8462403,DB,FLX,1239,2026-06-11,Erfurt Hbf,Germany,2026-06-11 12:10:00+00:00,intermediate,domestic,FlixTrain
8462400,DB,FLX,1239,2026-06-11,Fulda,Germany,2026-06-11 13:21:00+00:00,intermediate,domestic,FlixTrain
8462397,DB,FLX,1239,2026-06-11,Frankfurt(Main)Süd,Germany,2026-06-11 14:19:00+00:00,intermediate,domestic,FlixTrain
8462402,DB,FLX,1239,2026-06-11,Darmstadt Hbf,Germany,2026-06-11 14:37:00+00:00,intermediate,domestic,FlixTrain


Exporting the data as an intermediate output

In [16]:
export_data = input("Export intermediate data to parquet? (y/n): ")

if export_data.lower() == "y":
    os.makedirs(intermediate_outputs_dir, exist_ok=True)

    data_chuuchuu.to_parquet(f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_terminus.parquet")